# core

> `core` contains the `Unknown` class — a pseudonymisation/anonymisation engine for consistent cross-table key replacement and column masking, with Fernet-encrypted storage of mappings.

In [ ]:
#| default_exp core

I've done some prior exploration here's what I propose. 

**Project: `unknown`** — a pseudonymisation/anonymisation package built with nbdev (fast.ai style). The agreed design:

- **One class** (e.g. `Anonymiser`) with shared state across tables
- **`key_maps`**: `{group_id: {original_val: pseudonym}}` — for consistent cross-table key pseudonymisation
- **`vault`**: stores originals of masked columns so they can be restored
- **Config per table** using key groups like `id1`, `id2`, etc.:
  ```python
  tables = {
      'patients': {'id1': 'mid',       'mask': ['name', 'dob']},
      'visits':   {'id1': 'mid',       'id2': 'visit_code'},
      'doctors':  {'id2': 'doc_id',    'mask': ['name']},
  }
  ```
- **Lookup table approach** (random UUIDs, stored in a dict) — simple and reversible

The nbdev project is already scaffolded, and we're in `00_core.ipynb` which currently just has `def foo(): pass`. 

## Setup & data

In [ ]:
#| eval: false
import httpx
from pathlib import Path

base = "https://raw.githubusercontent.com/neo4j-contrib/northwind-neo4j/refs/heads/master/data"
files = ["customers.csv", "orders.csv", "employees.csv"]
dest = Path("../data")
dest.mkdir(exist_ok=True)

for f in files:
    (dest/f).write_bytes(httpx.get(f"{base}/{f}").content)
    print(f"✓ {f}")

✓ customers.csv
✓ orders.csv
✓ employees.csv


In [ ]:
#| eval: false
import pandas as pd
for f in files:
    df = pd.read_csv(dest/f,on_bad_lines='skip' )
    print(f"\n{f}: {df.shape}")
    print(df.columns.tolist())
    display(df.head(2))


customers.csv: (67, 11)
['customerID', 'companyName', 'contactName', 'contactTitle', 'address', 'city', 'region', 'postalCode', 'country', 'phone', 'fax']


,customerID,companyName,contactName,contactTitle,address,city,region,postalCode,country,phone,fax
0,ALFKI,Alfreds Futterkiste,Maria Anders,Sales Representative,Obere Str. 57,Berlin,NaN,12209,Germany,030-0074321,030-0076545
1,ANATR,Ana Trujillo Emparedados y helados,Ana Trujillo,Owner,Avda. de la Constitución 2222,México D.F.,NaN,05021,Mexico,(5) 555-4729,(5) 555-3745



orders.csv: (654, 14)
['orderID', 'customerID', 'employeeID', 'orderDate', 'requiredDate', 'shippedDate', 'shipVia', 'freight', 'shipName', 'shipAddress', 'shipCity', 'shipRegion', 'shipPostalCode', 'shipCountry']


,orderID,customerID,employeeID,orderDate,requiredDate,shippedDate,shipVia,freight,shipName,shipAddress,shipCity,shipRegion,shipPostalCode,shipCountry
0,10248,VINET,5,1996-07-04 00:00:00.000,1996-08-01 00:00:00.000,1996-07-16 00:00:00.000,3,32.38,Vins et alcools Chevalier,59 rue de l'Abbaye,Reims,NaN,51100,France
1,10249,TOMSP,6,1996-07-05 00:00:00.000,1996-08-16 00:00:00.000,1996-07-10 00:00:00.000,1,11.61,Toms Spezialitäten,Luisenstr. 48,Münster,NaN,44087,Germany



employees.csv: (9, 18)
['employeeID', 'lastName', 'firstName', 'title', 'titleOfCourtesy', 'birthDate', 'hireDate', 'address', 'city', 'region', 'postalCode', 'country', 'homePhone', 'extension', 'photo', 'notes', 'reportsTo', 'photoPath']


,employeeID,lastName,firstName,title,titleOfCourtesy,birthDate,hireDate,address,city,region,postalCode,country,homePhone,extension,photo,notes,reportsTo,photoPath
0,1,Davolio,Nancy,Sales Representative,Ms.,1948-12-08 00:00:00.000,1992-05-01 00:00:00.000,507 - 20th Ave. E. Apt. 2A,Seattle,WA,98122,USA,(206) 555-9857,5467,0x151C2F00020000000D000E0014002100FFFFFFFF4269...,Education includes a BA in psychology from Col...,2.0,http://accweb/emmployees/davolio.bmp
1,2,Fuller,Andrew,"Vice President, Sales",Dr.,1952-02-19 00:00:00.000,1992-08-14 00:00:00.000,908 W. Capital Way,Tacoma,WA,98401,USA,(206) 555-9482,3457,0x151C2F00020000000D000E0014002100FFFFFFFF4269...,Andrew received his BTS commercial in 1974 and...,NaN,http://accweb/emmployees/fuller.bmp


## Table configuration

In [ ]:
#| eval: false
tables = {
    'customers': {
        'id1': 'customerID',
        'mask': ['contactName', 'address', 'phone', 'companyName', 'fax', 'city']
    },
    'orders': {
        'id1': 'customerID',
        'id2': 'employeeID',
        'mask': ['shipName', 'shipAddress']
    },
    'employees': {
        'id2': 'employeeID',
        'mask': ['firstName', 'lastName', 'address', 'homePhone', 'birthDate', 'notes']
    },
}

Okay can we sketch this out? I started with these data structures and the North Wind dataset #Should I start building the Unknown class; We need English UK 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great config — that maps nicely to the design from the start notebook. A few thoughts on the sketch:

**Class name:** `Unknown` (as you suggested) — and we'll use UK English throughout (`anonymise`, `pseudonymise`, etc.)

**Core state:**
- `tables` — the config you've already written
- `key_maps` — `{group: {orig_val: pseudonym}}` e.g. `{'id1': {'ALFKI': 'uuid-xxx', ...}}`
- `vault` — stores original masked column values per table, so you can restore them

**Key methods to start with:**
1. `pseudonymise(name, df)` — takes a table name + DataFrame, pseudonymises keys and masks columns, returns the cleaned DataFrame
2. `restore(name, df)` — reverses the process

Does that match your thinking? If so, go ahead and stub out the class with `__init__` — just setting up those three attributes from the config. I'd suggest starting simple: take `tables` as the only `__init__` parameter.

## Building the `Unknown` class

In [ ]:
#| eval: false
class Unknown(object):
    def __init__(self, tables):
        self.tables = tables
        self.vault = {}
        self.key_maps = {}
    def encode(self, name,df): pass
    def decode(self, name,df): pass

#Simple start. Let's review.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Nice clean stub! A few notes:

- **`encode`/`decode`** — good naming choice, cleaner than `pseudonymise`/`restore`
- `(object)` — you can drop that, it's implicit in Python 3, but no harm keeping it
- You might want `#| export` above the class so nbdev exports it to your module

For `encode`, the logic will be roughly:
1. **Pseudonymise keys** — for each `id*` group in `tables[name]`, map original values → UUIDs (reusing existing mappings from `key_maps`)
2. **Vault & mask** — for each column in `mask`, stash the originals in `vault[name]`, then blank/redact them

Want to tackle step 1 first — the key pseudonymisation bit inside `encode`?

## Key pseudonymisation

In [ ]:
#| eval: false
for t in tables.keys():
    print("table is->", t)
    print(tables[t])

table is-> customers
{'id1': 'customerID', 'mask': ['contactName', 'address', 'phone', 'companyName', 'fax', 'city']}
table is-> orders
{'id1': 'customerID', 'id2': 'employeeID', 'mask': ['shipName', 'shipAddress']}
table is-> employees
{'id2': 'employeeID', 'mask': ['firstName', 'lastName', 'address', 'homePhone', 'birthDate', 'notes']}


In [ ]:
#| eval: false
ids = {k: v for k, v in tables['orders'].items() if k.startswith('id')}
ids

{'id1': 'customerID', 'id2': 'employeeID'}

In [ ]:
#| eval: false
customers = pd.read_csv(dest/"customers.csv", on_bad_lines='skip')
customers.sample(2)

,customerID,companyName,contactName,contactTitle,address,city,region,postalCode,country,phone,fax
14,ERNSH,Ernst Handel,Roland Mendel,Sales Manager,Kirchgasse 6,Graz,NaN,8010,Austria,7675-3425,7675-3426
25,KOENE,Königlich Essen,Philip Cramer,Sales Associate,Maubelstr. 90,Brandenburg,NaN,14776,Germany,0555-09876,NaN


In [ ]:
#| eval: false
unique_customers = customers['customerID'].unique()
unique_customers
import uuid
u = uuid.uuid4()

OK, I got as far as this. Before I continue, explain briefly:

1. why uuid4 is this a more secure id format
2. i assume i can from each uuid transform this into hex, int or other format? that would be the id itself

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good questions!

**1. Why uuid4?** It's generated from **cryptographically random** bits — there's no timestamp, MAC address, or sequence number baked in. That means you can't reverse-engineer any info about *when* or *where* it was created. With 122 random bits, the chance of a collision is astronomically small (you'd need to generate billions per second for decades). That's what makes it ideal for pseudonymisation — the fake ID reveals nothing about the original.

**2. Formats** — Yes, exactly. A UUID is just a 128-bit number. You can represent it as:
- `str(u)` → `'a3b2c1d4-...'` (the familiar hyphenated hex)
- `u.hex` → `'a3b2c1d4...'` (32 hex chars, no hyphens)
- `u.int` → a big integer
- `u.bytes` → 16 raw bytes

For your key maps, `str(u)` is probably the most readable choice. You could also use `u.hex` if you want something shorter and cleaner.

Ready to build that `{original: uuid}` mapping dict for the customerID column?

In [ ]:
#| eval: false
{i:str(uuid.uuid4()) for i in unique_customers[:3]}

{'ALFKI': '6572af35-e5a8-43ff-9e2d-04e2b47e2260',
 'ANATR': '405bbfdc-e22a-4a1d-987c-f4fdfb3bc709',
 'ANTON': '9a193352-eca5-4c15-9b6d-f5ba5394d04d'}

OK, let's rewrite this into the class. 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Nice — that comprehension is exactly the core logic. Now think about where it fits in `encode`:

1. Extract the id groups (you already know how: `k.startswith('id')`)
2. For each group, check if `self.key_maps[group]` already exists — if so, reuse it; if not, build it from the unique values
3. Replace the column values using the map

A hint for step 2–3: `df[col].map(mapping)` will swap values using a dict.

Have a go at writing the key pseudonymisation part of `encode` — just that bit, not the masking yet.

In [ ]:
#| eval: false
class Unknown(object):
    def __init__(self, tables):
        self.tables = tables
        self.vault = {}
        self.key_maps = {}
    def encode(self, name,df):
        # extract the ids
        tbl = self.tables[name]
        ids = {k: v for k, v in tbl.items() if k.startswith('id')}
        for group, col in ids.items():
            if group not in self.key_maps: self.key_maps[group] = {}
            uniques = df[col].dropna().unique()
            for u in uniques:
                if u not in self.key_maps[group]: self.key_maps[group][u] = str(uuid.uuid4())
    def decode(self, name,df): pass

I'm not super happy with my code so far, let's review if I can make this more readable

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Your logic is solid — it's just a bit dense. Two quick wins:

1. **`setdefault`** replaces the `if group not in` check:
   ```python
   km = self.key_maps.setdefault(group, {})
   ```
   This returns the existing dict or creates an empty one — one line instead of two.

2. **Inner loop** — once you have `km`, you can tighten it:
   ```python
   for val in df[col].dropna().unique():
       km.setdefault(val, str(uuid.uuid4()))
   ```
   Same pattern — only inserts if the key is missing.

That removes both `if` statements and makes the intent clearer. Give those two changes a try and see how it reads.

I've never used setdefault; could you explain this a bit more? Is this a recent Python addition?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠

Not recent at all — it's been in Python since version 2.0! It's just one of those dict methods that many people overlook.

Here's what it does:

```python
d = {}
d.setdefault('x', 42)   # 'x' not in d, so it sets d['x'] = 42 and returns 42
d.setdefault('x', 99)   # 'x' already in d, so it does nothing and returns 42
```

So it's a **get-or-insert** in one call. It always returns the value — either the existing one or the newly inserted default. That's what makes it perfect for building up dicts of dicts:

```python
km = self.key_maps.setdefault(group, {})
```

After this line, `km` is a reference to the inner dict — whether it already existed or was just created. So any changes you make to `km` are reflected in `self.key_maps[group]` too.

Does that make sense? Try it out in a quick test cell if you want to see it in action.

In [ ]:
#| eval: false
d = {}
d.setdefault('x', 42)   # 'x' not in d, so it sets d['x'] = 42 and returns 42
d.setdefault('x', 99)   # 'x' already in d, so it does nothing and returns 42

42

In [ ]:
#| eval: false
class Unknown(object):
    def __init__(self, tables):
        self.tables = tables
        self.vault = {}
        self.key_maps = {}
    def encode(self, name,df):
        df =df.copy()
        tbl = self.tables[name]
        ids = {k: v for k, v in tbl.items() if k.startswith('id')}
        for group, col in ids.items():
            km = self.key_maps.setdefault(group, {}) # will create or retreive; sourcery :) 
            for val in  df[col].dropna().unique(): km.setdefault(val, str(uuid.uuid4()))
            df[col] = df[col].map(km)
        return df 
    def decode(self, name,df): pass


Let's review this code iteration.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Looking much cleaner! Key improvements:

- **`df.copy()`** — smart addition, avoids mutating the original DataFrame
- **`setdefault`** — reads much more clearly now
- **`df[col].map(km)`** — does the actual swap, nice and concise
- **Returns `df`** — so you get the pseudonymised version back

One thing to be aware of: `map(km)` will turn any value **not** in the dict into `NaN` — including values that were already `NaN`. That's probably fine for ID columns, but worth knowing.

Want to test it on the customers table and see what comes out?

In [ ]:
#| eval: false
unk = Unknown(tables)
unk.encode('customers', customers).sample(2)

,customerID,companyName,contactName,contactTitle,address,city,region,postalCode,country,phone,fax
43,dbe44b29-a2da-4567-bdee-0d60263ace2c,Piccolo und mehr,Georg Pipps,Sales Manager,Geislweg 14,Salzburg,NaN,5020,Austria,6562-9722,6562-9723
7,ae62a9a1-946a-4dfa-baf9-9e65c3d8abd8,B's Beverages,Victoria Ashworth,Sales Representative,Fauntleroy Circus,London,NaN,EC2 5NT,UK,(171) 555-1212,NaN


OK, this seems to work. Let's add pseudonymise and mask methods and call it when encoding. Please review

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good refactor idea — separate concerns. Here's the shape I'd suggest:

**`_pseudonymise(self, name, df)`** — extracts the key-mapping logic you already have in `encode` (the `ids` loop). Returns the modified `df`.

**`_mask(self, name, df)`** — stores the original values of the `mask` columns in `self.vault[name]`, then replaces them with something (e.g. `'***'` or `None`). Returns the modified `df`.

**`encode`** becomes just:
```python
def encode(self, name, df):
    df = df.copy()
    df = self._pseudonymise(name, df)
    df = self._mask(name, df)
    return df
```

I'd use underscore-prefixed names (`_pseudonymise`, `_mask`) since they're internal — the public API is just `encode`/`decode`.

For `_mask`, the key question is: how do you want to store originals in the vault? One natural choice is `self.vault[name] = df[mask_cols].copy()` — stash the whole sub-DataFrame before blanking.

Have a go at writing those two methods!

## Column masking

In [ ]:
#| eval: false
class Unknown(object):
    def __init__(self, tables):
        self.tables = tables
        self.vault = {}
        self.key_maps = {}
    def encode(self, name,df):
        df = self._pseudonymise_keys(name, df)
        return self._mask_columns(name, df)
 
    def _pseudonymise_keys(self, name,df):
        df = df.copy()
        tbl = self.tables[name]
        ids = {k: v for k, v in tbl.items() if k.startswith('id')}
        for group, col in ids.items():
            km = self.key_maps.setdefault(group, {})
            for val in  df[col].dropna().unique(): km.setdefault(val, str(uuid.uuid4()))
            df[col] = df[col].map(km)
        return df
    
    def _mask_columns(self, name, df): pass     
    
    
    def decode(self, name,df): pass


In [ ]:
#| eval: false
unk = Unknown(tables)

unk._pseudonymise_keys('customers', customers).sample(10)

,customerID,companyName,contactName,contactTitle,address,city,region,postalCode,country,phone,fax
24,82108fe4-8ce1-434d-9912-a48aa05c69bd,Island Trading,Helen Bennett,Marketing Manager,Garden House Crowther Way,Cowes,Isle of Wight,PO31 7PJ,UK,(198) 555-8888,NaN
13,e6eec983-00c6-4037-b55c-611034515dee,Eastern Connection,Ann Devon,Sales Agent,35 King George,London,NaN,WX3 6FW,UK,(171) 555-0297,(171) 555-3373
18,c1d13160-cfc5-4223-b58f-502076f6933e,Furia Bacalhau e Frutos do Mar,Lino Rodriguez,Sales Manager,Jardim das rosas n. 32,Lisboa,NaN,1675,Portugal,(1) 354-2534,(1) 354-2535
57,2c49a502-ca04-436e-99e4-fdaa846b04bc,Toms Spezialitäten,Karin Josephs,Marketing Manager,Luisenstr. 48,Münster,NaN,44087,Germany,0251-031259,0251-035695
30,86daf560-f760-452b-8926-78bd1a8e564f,Let's Stop N Shop,Jaime Yorres,Owner,87 Polk St. Suite 5,San Francisco,CA,94117,USA,(415) 555-5938,NaN
42,b65cfdff-b2e0-46f5-98db-4c7f86f4fc0e,Pericles Comidas clásicas,Guillermo Fernández,Sales Representative,Calle Dr. Jorge Cash 321,México D.F.,NaN,05033,Mexico,(5) 552-3745,(5) 545-3745
5,ded853f9-8367-4904-91be-3bafae5ead54,Blauer See Delikatessen,Hanna Moos,Sales Representative,Forsterstr. 57,Mannheim,NaN,68306,Germany,0621-08460,0621-08924
66,8627dc11-0564-46f1-b487-5bbb410b98c8,Wolski Zajazd,Zbyszek Piestrzeniewicz,Owner,ul. Filtrowa 68,Warszawa,NaN,01-012,Poland,(26) 642-7012,(26) 642-7012
15,82ace702-dc3a-4a3a-8278-0d9f4fa9f786,Folk och fä HB,Maria Larsson,Owner,Åkergatan 24,Bräcke,NaN,S-844 67,Sweden,0695-34 67 21,NaN
36,ab8f49eb-5d66-415d-bf17-35a57e0443a0,Mère Paillarde,Jean Fresnière,Marketing Assistant,43 rue St. Laurent,Montréal,Québec,H1J 1C3,Canada,(514) 555-8054,(514) 555-8055


Let's use a hex truncated key for the mask and store in a dict across all dataframes. This way same names or addresses can be retrieved. Also let's allow nulls here not a problem to decode. I will address these in ids

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good thinking — a shared vault means if the same address appears in both `customers` and `orders`, it gets the same pseudonym. So `vault` becomes a flat lookup like `key_maps`:

- `self.vault = {}` — `{original_value: truncated_hex}` shared across all tables
- NaNs pass through untouched
- Something like `uuid.uuid4().hex[:12]` for the replacement (you pick the length)

The logic in `_mask_columns` would be very similar to `_pseudonymise_keys` — loop over the `mask` columns, build mappings in `self.vault`, then `df[col].map(self.vault)`. The key difference is you'd want to skip NaNs rather than mapping them.

Hint: `df[col].map(self.vault)` already leaves NaN as NaN, so that's handled for free.

Have a go at implementing `_mask_columns`?

In [ ]:
#| eval: false
class Unknown(object):
    def __init__(self, tables):
        self.tables = tables
        self.vault = {}
        self.key_maps = {}

    def encode(self, name, df):
        df = self._pseudonymise_keys(name, df)
        return self._mask_columns(name, df)

    def _remap(self, df, col, mapping, id_fn):
        for val in df[col].dropna().unique():
            mapping.setdefault(val, id_fn())
        df[col] = df[col].map(mapping)
        return df

    def _pseudonymise_keys(self, name, df):
        df = df.copy()
        tbl = self.tables[name]
        ids = {k: v for k, v in tbl.items() if k.startswith('id')}
        for group, col in ids.items():
            assert df[col].notna().all(), f"Null found in {col} for table {name}"
            km = self.key_maps.setdefault(group, {})
            df = self._remap(df, col, km, lambda: str(uuid.uuid4()))
        return df

    def _mask_columns(self, name, df):
        tbl = self.tables[name]
        for col in tbl.get('mask', []):
            df = self._remap(df, col, self.vault, lambda: uuid.uuid4().hex[:12])
        return df

    def decode(self, name, df): pass

OK, review code for now after latest changes; highlight these.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Nice iteration! Here's what changed and why it's good:

**`_remap` helper** — You extracted the common pattern (loop uniques → setdefault → map) into one method, with `id_fn` as the generator. Both `_pseudonymise_keys` and `_mask_columns` now use it. That's a solid DRY refactor.

**`assert` on key columns** — Smart. IDs shouldn't be null — catching that early avoids silent bugs downstream. Good call separating this concern from `_mask_columns` which is fine with nulls.

**Shared `self.vault`** — Flat dict across all tables, so `"Maria Anders"` gets the same hex whether it appears in customers or elsewhere. Exactly what you described.

**`hex[:8]`** — 8 hex chars = 32 bits ≈ 4 billion possibilities. Should be fine for this dataset, but worth considering if you'd want longer for production use.

One thing to watch: `_remap` is called from both methods, but `_pseudonymise_keys` does the `df.copy()` while `_mask_columns` doesn't — that's fine since `_pseudonymise_keys` always runs first in `encode`, but it's an implicit dependency. Something to keep in mind.

Want to test it end-to-end now?

In [ ]:
#| eval: false
unk = Unknown(tables)
for name in tables:
    df = pd.read_csv(dest/f"{name}.csv", on_bad_lines='skip')
    print(f"\n--- {name} ---")
    display(unk.encode(name, df).head(3))


--- customers ---


,customerID,companyName,contactName,contactTitle,address,city,region,postalCode,country,phone,fax
0,8f414e12-5226-41a0-b538-8f1433a36227,fcb01c97b103,99d0737545a0,Sales Representative,980923a97105,765534625ac9,NaN,12209,Germany,247d00afec5b,a35b45e97e9c
1,19be76d4-3a67-4114-bc18-c281702a92ca,6f9bffd7f85b,c47db0b858a8,Owner,927d2b8a6d2f,7734dec8e1ad,NaN,05021,Mexico,82c60cb68de9,a37de0195fbe
2,5199b8d0-4a36-49f7-8207-e355e72af2bd,0fa3101e241d,71cfa16fdf7f,Owner,8114f77b29b3,7734dec8e1ad,NaN,05023,Mexico,3aefaed7d23d,NaN



--- orders ---


,orderID,customerID,employeeID,orderDate,requiredDate,shippedDate,shipVia,freight,shipName,shipAddress,shipCity,shipRegion,shipPostalCode,shipCountry
0,10248,cd21310c-8ad1-497c-b86e-ad14af9e9958,fbeb38ef-97d1-4872-b6df-a9a19f6bdeb4,1996-07-04 00:00:00.000,1996-08-01 00:00:00.000,1996-07-16 00:00:00.000,3,32.38,92115b811676,c5e7ba796882,Reims,NaN,51100,France
1,10249,fc185abb-508b-4d71-90a4-cd6066c56716,7ffa4b9f-d40d-4ccf-a7ac-5a1609413446,1996-07-05 00:00:00.000,1996-08-16 00:00:00.000,1996-07-10 00:00:00.000,1,11.61,2b4e836e7061,d02781e99829,Münster,NaN,44087,Germany
2,10254,70855f6b-5896-42de-b726-272d73b458c8,fbeb38ef-97d1-4872-b6df-a9a19f6bdeb4,1996-07-11 00:00:00.000,1996-08-08 00:00:00.000,1996-07-23 00:00:00.000,2,22.98,8bac48a62768,8f4b0ec0de2b,Bern,NaN,3012,Switzerland



--- employees ---


,employeeID,lastName,firstName,title,titleOfCourtesy,birthDate,hireDate,address,city,region,postalCode,country,homePhone,extension,photo,notes,reportsTo,photoPath
0,9c95d553-9e5a-416d-97a2-cb57cc0b092c,4a4c8c375f46,2b32b63aa47a,Sales Representative,Ms.,bcea13e9f414,1992-05-01 00:00:00.000,f762d5ce0c0b,Seattle,WA,98122,USA,dd1a265928e4,5467,0x151C2F00020000000D000E0014002100FFFFFFFF4269...,de9be2d80bf4,2.0,http://accweb/emmployees/davolio.bmp
1,aef7eabe-02ef-409c-8bb5-bb29bee3fdf9,2f2acc73e3d7,841a7734e7ab,"Vice President, Sales",Dr.,e96443b49e7d,1992-08-14 00:00:00.000,d9a9f6c177ad,Tacoma,WA,98401,USA,24be59f9e04c,3457,0x151C2F00020000000D000E0014002100FFFFFFFF4269...,82ed0485db6d,NaN,http://accweb/emmployees/fuller.bmp
2,18ab2b5a-f530-4a51-9e0d-86cf1780efe4,bd489e562330,6aa3c968ac75,Sales Representative,Ms.,06403d5c506e,1992-04-01 00:00:00.000,08b084697e63,Kirkland,WA,98033,USA,eef0e11dfab7,3355,0x151C2F00020000000D000E0014002100FFFFFFFF4269...,a8b24ee99395,2.0,http://accweb/emmployees/leverling.bmp


In [ ]:
#| eval: false
unk.key_maps

{'id1': {'ALFKI': '8f414e12-5226-41a0-b538-8f1433a36227',
  'ANATR': '19be76d4-3a67-4114-bc18-c281702a92ca',
  'ANTON': '5199b8d0-4a36-49f7-8207-e355e72af2bd',
  'AROUT': 'a7763bb5-6615-4ad0-92af-f3dd6a006a2a',
  'BERGS': '17d168a5-5d4f-4ae2-a73e-75ed47aa826e',
  'BLAUS': 'b5c08fd3-3abf-4e88-b494-2eed884fa69c',
  'BOTTM': 'af7841d5-8a53-4a60-9cd9-17c9b7133498',
  'BSBEV': 'e624d7fe-7b04-4c25-8219-e6bcd6b0628b',
  'CACTU': '20398fa9-26d0-4f28-9644-3fa6b34f209d',
  'CENTC': '451e5acc-856c-45d5-afcb-20650400a83a',
  'CHOPS': '70855f6b-5896-42de-b726-272d73b458c8',
  'CONSH': 'a7610fc0-2ba3-410a-a1e2-5cd257ac82e5',
  'DRACD': '67b0c195-7163-4b3e-81d6-7cea8f33467d',
  'EASTC': '06c29a85-9d3d-492f-91aa-8b82976d191e',
  'ERNSH': '77f91632-6b0c-4c30-997d-1e4da588e7b6',
  'FOLKO': '45f983c7-6033-4ece-9a13-1d5ec40f2cd9',
  'FRANK': '42447782-b840-4491-8ef9-bdae1d083685',
  'FRANS': 'b4b8bf3e-4a21-4685-b563-7aaa3c54de1f',
  'FURIB': '4ee703be-4403-4ea0-824e-3a8f97af7f69',
  'GREAL': '1470689e-417

In [ ]:
#| eval: false
unk.vault

{'Maria Anders': '99d0737545a0',
 'Ana Trujillo': 'c47db0b858a8',
 'Antonio Moreno': '71cfa16fdf7f',
 'Thomas Hardy': 'afd115e8488c',
 'Christina Berglund': '46db1fe95ef3',
 'Hanna Moos': 'e42060ac33ab',
 'Elizabeth Lincoln': 'a46314fa291c',
 'Victoria Ashworth': '3b888df8c13d',
 'Patricio Simpson': '9ae7db3302bb',
 'Francisco Chang': 'a7f1bdccb773',
 'Yang Wang': '0fdec5e13850',
 'Elizabeth Brown': '3cfb87a2d3f7',
 'Sven Ottlieb': 'ca9e362ee660',
 'Ann Devon': '23842193e2fe',
 'Roland Mendel': '757d91515a72',
 'Maria Larsson': 'fe9622a6b042',
 'Peter Franken': '63ef284b80d4',
 'Paolo Accorti': '02f54e7b20e9',
 'Lino Rodriguez': 'e304639afc4b',
 'Howard Snyder': 'b3b6e2e10792',
 'Manuel Pereira': 'e8ecc0a053fb',
 'Carlos Hernández': '1d58da47c887',
 'Yoshi Latimer': 'de1a1c32a8d7',
 'Patricia McKenna': 'f091c20e67fb',
 'Helen Bennett': '22846234e096',
 'Philip Cramer': 'f37145dad156',
 'Annette Roulet': '53df4f089462',
 'Yoshi Tannamuri': 'f543647d597d',
 'John Steel': 'a4b1e6a9eec1',


Let's work on decoding this now, the inverse property. Requirement: I also would like something to store the encoded values. Now let's just not assume that these would be available within the class because likely this could be used at the end of a pipeline 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good thinking — in a real pipeline, encoding and decoding would likely happen in separate steps (maybe even separate machines). So you need two things:

1. **Persist the mappings** — `key_maps` and `vault` need to be saveable/loadable (JSON is natural since they're just nested dicts of strings)
2. **`decode`** — invert the mappings (`{pseudonym: original}`) and `.map()` back

A couple of design questions for you:

- For storage, are you happy with a simple `save(path)` / `load(path)` pair using JSON? Or do you want something fancier?
- Should `decode` work from the in-memory state (if available) **and** from a loaded file? Or only from file?

For a start, simple save and restore is fine. But given this is sensitive data, I would like to discuss some encryption at rest.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good instinct — those mappings are essentially the keys to de-anonymise everything, so they're the most sensitive artefact in the whole pipeline.

For encryption at rest, the simplest solid option is **Fernet** from the `cryptography` library:
- Symmetric encryption (one key encrypts and decrypts)
- AES-128-CBC under the hood, with HMAC authentication
- Password → key derivation via PBKDF2 is built in

The flow would be:
1. **Save**: JSON serialize → encrypt with Fernet → write bytes to file
2. **Load**: read bytes → decrypt with Fernet → JSON deserialize

The main design choice: do you want a **password-based** key (user supplies a passphrase each time) or a **generated key** stored separately (e.g. in an env var or key file)?

## Encryption at rest

In [ ]:
#| eval: false
from cryptography.fernet import Fernet

key = Fernet.generate_key()    # bytes, save this somewhere safe
f = Fernet(key)

encrypted = f.encrypt(b"hello")  # bytes in → bytes out
f.decrypt(encrypted)             # b"hello"

encrypted

b'gAAAAABprvvhrUvA4K0t4ZuXLaWhE-KTCTwk7v7Yp5SZzkpto9RrKNS_Mqbi2sMSFzpNufINWPB9W3D5h0uZgl16DtEcPLocsA=='

Assume I use Fernet. Let's rethink the class to have this option to generate the key and store it first. I want it outside its state like a static method maybe

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good call — keeping the key outside instance state means the `Unknown` object itself is never holding the secret. Clean separation.

The shape would be:

- **`Unknown.generate_key()`** — `@staticmethod`, returns a Fernet key (bytes). User stores it however they like (env var, file, vault)
- **`save(path, key)`** — serialise `key_maps` + `vault` to JSON, encrypt with the provided key, write to file
- **`load(path, key)`** — read file, decrypt, deserialise, return a new `Unknown` instance (or populate `key_maps`/`vault`)

So the key is always **passed in** at save/load time, never stored on `self`. The user is responsible for keeping it safe.

Want to start by adding the `generate_key` staticmethod?

In [ ]:
#| eval: false
class Unknown(object):
    def __init__(self, tables):
        self.tables = tables
        self.vault = {}
        self.key_maps = {}

    def encode(self, name, df):
        df = self._pseudonymise_keys(name, df)
        return self._mask_columns(name, df)

    def _remap(self, df, col, mapping, id_fn):
        for val in df[col].dropna().unique():
            mapping.setdefault(str(val), id_fn()) # add a new key if not found to val (id_fn() return)
        df[col] = df[col].astype(str).map(mapping)
        return df

    def _pseudonymise_keys(self, name, df):
        df = df.copy()
        
        tbl = self.tables[name]
        ids = {k: v for k, v in tbl.items() if k.startswith('id')}
        for group, col in ids.items():
            assert df[col].notna().all(), f"Null found in {col} for table {name}"
            assert pd.api.types.is_string_dtype(df[col]), f"Column {col} must be string type, got {df[col].dtype}"
            km = self.key_maps.setdefault(group, {})
            df = self._remap(df, col, km, lambda: str(uuid.uuid4()))
        return df

    def _mask_columns(self, name, df):
        tbl = self.tables[name]
        for col in tbl.get('mask', []):
            df = self._remap(df, col, self.vault, lambda: uuid.uuid4().hex[:12])
        return df

    def decode(self, name, df): pass

    @staticmethod
    def generate_key(path=None):
        from cryptography.fernet import Fernet
        key = Fernet.generate_key()
        if path:
            Path(path).write_bytes(key)
        return key
    
    @staticmethod
    def load_key(path=None, env_var=None):
        if path: return Path(path).read_bytes()
        if env_var:
            import os
            return os.environ[env_var].encode()
        raise ValueError("Provide either path or env_var")



In [ ]:
#| eval: false
# key = Unknown.generate_key()
# key.decode()

let's discuss where to include the fernet key load. I would argue at init. the user should have one set prior to using this 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

That's a reasonable argument — it ensures encryption is always available. But consider the trade-off:

**Key at `__init__`** — enforces that encryption is set up from the start. `save()`/`load()` can just use `self._key` internally. Clean API.

**Key at `save()`/`load()` only** — someone who just wants to `encode` a DataFrame in memory (no persistence) doesn't need to deal with keys at all. Lower barrier for simple use cases.

A middle ground: make it **optional** at init, but **required** for save/load. Something like:

```python
def __init__(self, tables, key=None):
```

If `key` is set, `save()` just works. If not, `save()` raises a clear error. That way casual usage stays simple, but the pipeline path is solid.

What's your preference?

In [ ]:
#| eval: false
from fastcore.basics import patch
import json

@patch
def save(self: Unknown, path, key):
    """Encrypt and save key_maps + vault to file."""
    from cryptography.fernet import Fernet
    data = json.dumps({'key_maps': self.key_maps, 'vault': self.vault})
    encrypted = Fernet(key).encrypt(data.encode())
    Path(path).write_bytes(encrypted)

@patch(cls_method=True)
def load(cls: Unknown, path, key, tables):
    """Load encrypted mappings and return a new Unknown instance."""
    from cryptography.fernet import Fernet
    encrypted = Path(path).read_bytes()
    data = json.loads(Fernet(key).decrypt(encrypted))
    unk = cls(tables)
    unk.key_maps, unk.vault = data['key_maps'], data['vault']
    return unk

In [ ]:
#| eval: false
bad_key = Unknown.generate_key()
try:
    Unknown.load(dest/"mappings.enc", bad_key, tables)
    assert False, "Should have raised"
except Exception as e:
    print(f"✓ Correctly rejected: {type(e).__name__}")

✓ Correctly rejected: InvalidToken


In [ ]:
#| eval: false
@patch
def decode(self: Unknown, name, df):
    """Reverse pseudonymisation and masking for key columns and masked columns"""
    df = df.copy()
    tbl = self.tables[name]
    # Inverse key_maps for each id-group
    ids = {k: v for k, v in tbl.items() if k.startswith('id')}
    for group, col in ids.items():
        km = self.key_maps[group]
        inv_km = {v: k for k, v in km.items()}
        df[col] = df[col].map(inv_km)
    # Masked columns
    inv_vault = {v: k for k, v in self.vault.items()}
    for col in tbl.get('mask', []):
        df[col] = df[col].map(inv_vault)
    return df

In [ ]:
#| eval: false
import os

# 1. Generate key and store in env var
key = Unknown.generate_key()
os.environ['UNKNOWN_KEY'] = key.decode()
loaded_key = Unknown.load_key(env_var='UNKNOWN_KEY')
assert loaded_key == key
print("✓ Key round-tripped via env var")

# 2. Encode all tables
unk = Unknown(tables)
originals, encoded = {}, {}
for name in tables:
    df = pd.read_csv(dest/f"{name}.csv", on_bad_lines='skip')
    for k, v in tables[name].items():
        if k.startswith('id'): df[v] = df[v].astype(str)
    originals[name] = df
    encoded[name] = unk.encode(name, originals[name])
    print(f"✓ Encoded {name}")

# 3. Save, load with env key, decode and compare
unk.save(dest/"mappings.enc", loaded_key)
unk2 = Unknown.load(dest/"mappings.enc", loaded_key, tables)
for name in tables:
    decoded = unk2.decode(name, encoded[name])
    pd.testing.assert_frame_equal(decoded, originals[name])
    print(f"✓ Decoded {name} matches original")

✓ Key round-tripped via env var
✓ Encoded customers
✓ Encoded orders
✓ Encoded employees
✓ Decoded customers matches original
✓ Decoded orders matches original
✓ Decoded employees matches original


## Final class & round-trip test

In [ ]:
#| export
#| export
import uuid, json
from pathlib import Path
from cryptography.fernet import Fernet
import pandas as pd


class Unknown:
    """Pseudonymise and mask DataFrames consistently across related tables."""
    def __init__(self, tables):
        self.tables,self.vault,self.key_maps = tables,{},{}

    def encode(self, name, df):
        """Pseudonymise keys and mask columns for table `name`, returning new DataFrame."""
        df = self._pseudonymise_keys(name, df)
        return self._mask_columns(name, df)

    def decode(self, name, df):
        """Reverse `encode` — restore original keys and masked values."""
        df = df.copy()
        tbl = self.tables[name]
        ids = {k: v for k, v in tbl.items() if k.startswith('id')}
        for group, col in ids.items():
            inv = {v: k for k, v in self.key_maps[group].items()}
            df[col] = df[col].map(inv)
        inv_vault = {v: k for k, v in self.vault.items()}
        for col in tbl.get('mask', []):
            df[col] = df[col].map(inv_vault)
        return df

    def _remap(self, df, col, mapping, id_fn):
        """Replace `col` values via `mapping`, adding new entries with `id_fn`."""
        for val in df[col].dropna().unique():
            mapping.setdefault(str(val), id_fn())
        df[col] = df[col].astype(str).map(mapping)
        return df

    def _pseudonymise_keys(self, name, df):
        """Replace id columns with consistent UUIDs via `key_maps`."""
        df = df.copy()
        tbl = self.tables[name]
        ids = {k: v for k, v in tbl.items() if k.startswith('id')}
        for group, col in ids.items():
            assert df[col].notna().all(), f"Null found in {col} for table {name}"
            assert pd.api.types.is_string_dtype(df[col]), f"Column {col} must be string type, got {df[col].dtype}"
            km = self.key_maps.setdefault(group, {})
            df = self._remap(df, col, km, lambda: str(uuid.uuid4()))
        return df

    def _mask_columns(self, name, df):
        """Replace mask columns with truncated hex tokens via shared `vault`."""
        tbl = self.tables[name]
        for col in tbl.get('mask', []):
            df = self._remap(df, col, self.vault, lambda: uuid.uuid4().hex[:12])
        return df

    def save(self, path, key):
        """Encrypt and save `key_maps` + `vault` to `path` using Fernet `key`."""
        data = json.dumps({'key_maps': self.key_maps, 'vault': self.vault})
        Path(path).write_bytes(Fernet(key).encrypt(data.encode()))

    @classmethod
    def load(cls, path, key, tables):
        """Load encrypted mappings from `path` and return a new `Unknown` instance."""
        data = json.loads(Fernet(key).decrypt(Path(path).read_bytes()))
        unk = cls(tables)
        unk.key_maps, unk.vault = data['key_maps'], data['vault']
        return unk

    @staticmethod
    def generate_key(path=None):
        """Generate a Fernet key, optionally saving to `path`."""
        key = Fernet.generate_key()
        if path: Path(path).write_bytes(key)
        return key

    @staticmethod
    def load_key(path=None, env_var=None):
        """Load a Fernet key from `path` or `env_var`."""
        if path: return Path(path).read_bytes()
        if env_var:
            import os
            return os.environ[env_var].encode()
        raise ValueError("Provide either path or env_var")

In [ ]:
#| eval: false
%load_ext autoreload
%autoreload 2

In [ ]:
#| eval: false
from unknown.core import Unknown

Let's see if we can commit the latest changes 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Let's start by checking what's changed:

```bash
!git status
```

Run that and we'll see what needs staging. You'll probably also want to run `!nbdev_export` first if you haven't already, to make sure the module file is in sync with the notebook.

In [ ]:
#| eval: false
!git status

On branch master
Your branch is ahead of 'origin/master' by 1 commit.
  (use "git push" to publish your local commits)

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../README.md
	modified:   00_core.ipynb
	modified:   index.ipynb

no changes added to commit (use "git add" and/or "git commit -a")


ok i need to publish this package to pypi

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

A few things to sort out before publishing. First, let's check your package settings:

```bash
!cat settings.ini
```

This will show us the metadata (version, description, author, etc.) that PyPI needs. Run that and we'll review what needs updating.

In [ ]:
! ls ../

LICENSE      README.md	data  pyproject.toml  unknown.egg-info
MANIFEST.in  _proc	nbs   unknown


what is the setting file you are looking for?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Ah sorry — I was looking for `settings.ini` (the older nbdev convention), but your project has `pyproject.toml` instead. That's the one we need. Let's look at it:

```bash
!cat ../pyproject.toml
```

In [ ]:
!cat ../pyproject.toml

[build-system]
requires = ["setuptools>=64"]
build-backend = "setuptools.build_meta"

[project]
name = "unknown"
dynamic = ["version"]
description = "Pseudonymisation/anonymisation engine with encrypted mapping storage"
readme = "README.md"
requires-python = ">=3.10"
license = {text = "Apache-2.0"}
authors = [{name = "frangs", email = "FGiordano-Silva@lambeth.gov.uk"}]
keywords = ['nbdev']
classifiers = [
    "Programming Language :: Python :: 3",
    "Programming Language :: Python :: 3 :: Only",
]
dependencies = ["cryptography", "pandas"]

[project.urls]
Repository = "https://github.com/frangs/unknown"
Documentation = "https://frangs.github.io/unknown/"

[project.entry-points.nbdev]
unknown = "unknown._modidx:d"

[tool.setuptools.dynamic]
version = {attr = "unknown.__version__"}

[tool.setuptools.packages.find]
include = ["unknown"]

[tool.nbdev]


In [ ]:
!pip index versions unknown

ERROR: No matching distribution found for unknown


ok what are we missing here. dependencis and description added

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

The metadata looks good. Two things to check before building:

1. **Version** — your `pyproject.toml` reads version from `unknown.__version__`. Let's check it's set:

```bash
!cat ../unknown/__init__.py
```

2. **Package name** — `"unknown"` is extremely generic. PyPI might reject it or it could clash later. Have you considered something more specific like `unknown-anon` or `pseudonymiser`?

Run that `cat` and we'll see where the version stands.

In [ ]:
!cat ../unknown/__init__.py

__version__ = "0.0.1"


ok what next steps i belive ive setup my pypi token also

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good — version is set and token is ready. The steps are:

1. **Build** the distribution:
   ```bash
   !cd .. && python -m build
   ```

2. **Upload** to PyPI:
   ```bash
   !cd .. && twine upload dist/*
   ```

You may need to install `build` and `twine` first if they're not already available:
```bash
!pip install build twine
```

Start with the install and build step, and we'll see if it compiles cleanly.

i assume the built and all of the above are run in the project root folder where pyproject is?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes, exactly — `python -m build` and `twine upload` both expect to be run from the directory containing `pyproject.toml`. That's why the commands I gave used `cd ..` — since your notebook is in the `nbs/` subfolder.

So from a terminal it would just be:

```bash
cd /path/to/unknown   # where pyproject.toml lives
python -m build
twine upload dist/*
```

Or from the notebook, keep using `!cd .. && ...` to run from the project root.